# Lab 1: Text Processing, Sentiment and Topics with Classical NLP

**Time:** about 75 minutes

Read a few of these messages and you'll see what you're dealing with:

- *"Caller reports: Good morning, I submitted my maternity benefit claim in March and I have still not been paid."*
- *"Shikamoo, Tupo wafanyaazi 2 hakuna hata mmoja mwenye michango tangu Mei ... Nisaidieni tafadhli ..."*
- *"i have two member numbers from two employers, pls merge them"*

English, Swahili, both at once, typing errors, no capital letters, an agent's summary of a phone call. Real text,
in other words.

### Why text needs special treatment

Every model so far has worked on a table of numbers. Text isn't a table: one message is 5 words, the next is 80, and
the same idea can be written in a hundred ways. **Natural language processing (NLP)** is the set of techniques for
turning text into numbers a model can learn from, and then making sense of what it learns.

"Classical" NLP means the methods that came before the large language models of Days 7 to 9. They're fast, cheap, run
on any laptop, and are easy to explain, which is why they're still widely used.

In this lab you'll:

1. clean and **tokenise** the text (split it into words) in a way that works for both languages,
2. build a sentiment classifier with TF-IDF (term frequency-inverse document frequency) and logistic regression, a
   combination that's still hard to beat,
3. learn word embeddings (Word2Vec) from the Fund's own messages,
4. find the topics in negative feedback and watch them change over time.

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **NLP** | natural language processing: getting computers to work with human language |
| **token** | one unit of text after splitting, usually a word |
| **tokenise** | split a message into tokens |
| **stop words** | very common words (*the, and, na, ya*) that carry little meaning on their own |
| **stemming** | cutting words down to a root (*paying, paid → pay*); works for English only |
| **regular expression** | a pattern for finding text, e.g. "CLM- followed by 4 digits" |
| **sentiment** | whether a message is negative, neutral or positive |
| **vocabulary** | the list of all the words the model knows |
| **TF-IDF** | a way to turn a message into numbers, where words that are distinctive for that message count most |
| **n-gram** | a run of n words; a 2-gram (bigram) is a word pair like *not paid* |
| **word embedding** | a list of numbers for each word, where words used in similar ways get similar numbers |
| **Word2Vec** | a method that learns word embeddings from which words appear near each other |
| **topic model** | a method that finds groups of words that tend to appear together (themes) |
| **NMF** | non-negative matrix factorisation: the topic-model method used here |
| **F1 score** | one number combining precision and recall; *macro* F1 averages it over the classes |

## Where this lab fits

**What you've done so far.** For five days every model has worked on numbers: claim amounts, contribution histories,
payment features. On Day 2 you learned to classify (irregular or not), on Day 3 to train neural networks, and on Day 5
to find patterns with and without labels.

**What you'll do now.** You'll work with what members actually write: about 6,200 messages in English, Swahili and a
mix of both. You'll clean and split the text into words, turn it into numbers with TF-IDF, train a sentiment
classifier and read its reasoning, learn word embeddings from the Fund's own messages, and find the themes in negative
feedback as they change over time.

**Why it matters.** Members' messages are the most direct evidence of what's going wrong: delayed benefits, employers
not remitting, a new app that won't let people log in. Nobody can read them all. Tools that do the first pass let the
Fund see problems within weeks instead of months.

**What happens next.** Lab 2 replaces word counts with a transformer model (BERT) that understands words in context,
and uses it to route complaints to the right team. Lab 3 pulls facts (names, member numbers, dates, amounts) out of
officers' notes. Days 7-9 then move to large language models.

### Theory: three ways to turn text into numbers

Every NLP method has to answer one question first: how do you turn a message into numbers? Today you'll meet the three
main answers, in the order they were invented:

| Approach | Each message becomes | Knows that *pension* ≈ *pensheni*? | Understands context ("not paid")? | Where |
|---|---|---|---|---|
| **Bag of words / TF-IDF** | one number per vocabulary word | no | only through word pairs | Section 3 |
| **Word embeddings** (Word2Vec) | the average of its words' learned vectors | yes, if they're used alike | no: one vector per word | Section 4 |
| **Contextual embeddings** (transformers) | vectors that depend on the whole sentence | yes | yes | Lab 2 |

Each step up captures more meaning, and costs more data, computing power and explaining. The first is still the right
starting point: it's fast, it runs anywhere, and you can read exactly why it made each prediction.

## 0. Set up today's environment (run this cell first)

You don't need to create a folder or an environment by hand today. The cell below does it for you:

1. It finds the environment you created on Day 1 (`DAY1/Labs/work/fund-ai/.venv`). If it isn't there, it creates it.
2. It installs today's libraries into it from ``requirements.txt``. The first time takes a few minutes; after that, seconds.
3. If this machine has an NVIDIA GPU (graphics processing unit), it switches PyTorch to its GPU build, so training runs much faster.
4. It connects the environment to Jupyter as **Python (fund-ai)**, and tells you if this notebook isn't using it yet.

It's safe to run again at any time. If it prints instructions to switch the kernel, follow them and run it once more.

You don't need to read the code in this cell. It's housekeeping, not part of today's lesson.

In [ ]:
# ---- Setup for Day 6. Safe to run more than once. -------------------------------------------------------------
COURSE_ENV = None        # None = use the Day 1 environment. Or set a folder, e.g. Path.home() / "fund-ai-venv"
REQUIREMENTS = "requirements.txt"
EXTRAS = ['spacy:en_core_web_sm']

import os, shutil, subprocess, sys
from pathlib import Path

labs = Path.cwd()
root = next((p for p in [labs, *labs.parents] if (p / "DAY1").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the course folder (the one that contains DAY1, DAY2, ...).")
env = Path(COURSE_ENV) if COURSE_ENV else root / "DAY1" / "Labs" / "work" / "fund-ai" / ".venv"
py = env / ("Scripts/python.exe" if os.name == "nt" else "bin/python")


def run(*cmd, quiet=False):
    """Run a command; show its output only if it fails."""
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.returncode != 0 and not quiet:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if "path" in (r.stderr + r.stdout).lower() and os.name == "nt":
            print("\nIf the error mentions a path that is too long: Windows 'long paths' are switched off on this machine.\n"
                  "Ask ICT to enable them, or set COURSE_ENV above to a short folder, e.g. Path.home() / 'fund-ai-venv'.")
        raise RuntimeError("A setup step failed. The details are printed above.")
    return r


# 1. The course environment
if py.exists():
    print(f"- course environment found: {env}")
else:
    print(f"- creating the course environment (first time only): {env}")
    run(getattr(sys, "_base_executable", sys.executable), "-m", "venv", env)
    run(py, "-m", "pip", "install", "-q", "--upgrade", "pip")

# 2. Today's libraries
req = labs / REQUIREMENTS
print(f"- installing today's libraries from {REQUIREMENTS} ...")
run(py, "-m", "pip", "install", "-q", "-r", req)

# 3. Use the GPU if this machine has an NVIDIA one
if "torch" in req.read_text() and shutil.which("nvidia-smi"):
    if run(py, "-c", "import torch; print(torch.cuda.is_available())").stdout.strip() != "True":
        version = run(py, "-c", "import torch; print(torch.__version__.split('+')[0])").stdout.strip()
        print(f"- NVIDIA GPU found: switching PyTorch {version} to its GPU build (about 2.5 GB, first time only) ...")
        run(py, "-m", "pip", "uninstall", "-y", "-q", "torch")
        run(py, "-m", "pip", "install", "-q", f"torch=={version}", "--index-url", "https://download.pytorch.org/whl/cu124",
            "--extra-index-url", "https://pypi.org/simple")
    print("- PyTorch will use the GPU")

# 4. Anything else today needs
for extra in EXTRAS:
    kind, name = extra.split(":", 1)
    if kind == "spacy" and run(py, "-c", f"import {name}", quiet=True).returncode != 0:
        print(f"- downloading the spaCy model {name} ...")
        run(py, "-m", "spacy", "download", name)
    if kind == "ollama":
        import json, urllib.request
        try:
            tags = json.load(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3))["models"]
            have = {m["name"] for m in tags}
            wanted = [m for m in name.split(",") if m]
            missing = [m for m in wanted if m not in have]
            print(f"- Ollama is running with {len(have)} model(s)" + (f"; still to pull: {', '.join(missing)}" if missing else ""))
            for m in missing:
                print(f"    in a terminal: ollama pull {m}")
        except OSError:
            print("- Ollama isn't running. Start the Ollama app (or run `ollama serve` in a terminal) before the LLM parts.")

# 5. Connect the environment to Jupyter
run(py, "-m", "ipykernel", "install", "--user", "--name", "fund-ai", "--display-name", "Python (fund-ai)")

# 6. Is this notebook using it?
if Path(sys.prefix).resolve() == env.resolve():
    print("\nReady. This notebook is running in the course environment. Carry on with the next cell.")
else:
    print("\nAlmost there. This notebook is running a different Python:")
    print(f"    {sys.executable}")
    print("Switch it to the course environment, then run this cell once more:")
    print("    1. Click the kernel name in the top right of the notebook, then 'Select Another Kernel'.")
    print("    2. Choose 'Jupyter Kernel' -> 'Python (fund-ai)'.")
    print("       (Or press Ctrl+Shift+P, run 'Python: Select Interpreter', choose 'Enter interpreter path...'")
    print("        and paste the path below, the same way you did on Day 1.)")
    print(f"    {py}")

## Imports

The tools for today, grouped by job:

- **cleaning text:** `re` (regular expressions, for finding patterns such as claim numbers) and NLTK's
  `wordpunct_tokenize` (splits text into words and punctuation);
- **counting:** `Counter`, which counts how often each item appears;
- **turning text into numbers:** scikit-learn's `TfidfVectorizer`, plus its built-in list of English stop words;
- **modelling:** `LogisticRegression`, `train_test_split`, `Pipeline`, and `NMF` for topics;
- **measuring:** `classification_report` and `f1_score`.

`display.max_colwidth` lets pandas show long messages in full.

In [ ]:
import os
import re                                  # regular expressions: find and replace patterns in text
import warnings
from collections import Counter            # counts how often each item appears

# Skip a slow CPU-core check on Windows 11, and hide library notices.
os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count() - 1)
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.tokenize import wordpunct_tokenize           # splits text into words and punctuation

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.decomposition import NMF

pd.set_option("display.max_colwidth", 140)     # show up to 140 characters of each message
pd.set_option("display.width", 170)
SEED = 42

> **Where we are.** The text tools are loaded. First, a look at the messages and how the staff labelled them.

## 1. The feedback

`member_feedback.csv` has one row per message members sent the Fund, from several channels (call centre, WhatsApp,
email, social media). Each has been labelled by staff with a `category` (what it's about) and a `sentiment` (negative,
neutral or positive), and tagged with its `language` (`en`, `sw` or `mixed`).

`sample(8, random_state=1)` shows 8 random rows, the same 8 every time.

In [ ]:
fb = pd.read_csv("member_feedback.csv", parse_dates=["received_at"])
print(fb.shape)
fb.sample(8, random_state=1)[["channel", "language", "text", "category", "sentiment"]]

Three quick charts: how many messages there are in each category, each sentiment, and each language. Always look at
how the labels are spread before you model them; a rare class needs special attention later.

In [ ]:
fb["category"].value_counts().sort_values().plot(kind="barh", color="#2a78d6")     # horizontal bars
plt.title("Category")
plt.show()

In [ ]:
fb["sentiment"].value_counts().plot(kind="bar", color=["#e34948", "#b5b4ad", "#1baf7a"])
plt.title("Sentiment")
plt.xticks(rotation=0)       # keep the labels horizontal
plt.show()

In [ ]:
fb["language"].value_counts().plot(kind="bar", color="#2a78d6")
plt.title("Language")
plt.xticks(rotation=0)
plt.show()

Mostly negative, which is normal for a complaints channel: people rarely write in to say things are fine. About a
third of messages are in Swahili and a fifth mix both languages.

Check for duplicates before modelling. The same message logged twice would end up on both sides of a train/test split,
and the model would get credit for remembering it. `duplicated(subset=["text"])` is True for every repeat of a message
already seen; `drop_duplicates` keeps only the first copy, and `reset_index(drop=True)` renumbers the rows 0, 1, 2 ...

In [ ]:
n_duplicates = fb.duplicated(subset=["text"]).sum()
print(n_duplicates, "duplicate messages")

fb = fb.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(len(fb), "messages left")

> **Where we are.** You've seen the mix of categories, sentiments and languages, and removed duplicate messages so none
> can appear on both sides of a split. Now the text itself needs preparing.

### Theory: why cleaning text is a set of choices, not a recipe

Every cleaning step throws something away, and what's safe to throw away depends on the task:

| Step | Throws away | Safe for sentiment? |
|---|---|---|
| lower case | capital letters | usually yes ("PAID" and "paid" mean the same) |
| remove stop words | very common words | yes, **except negations**: "not paid" ≠ "paid" |
| replace claim and member numbers | the exact number | yes: the fact that a claim is mentioned matters, not which one |
| stemming | word endings | in English often yes; in Swahili it damages words |
| remove punctuation | `!`, `?` | mostly; "!!!" can carry feeling, but it's a small loss |

For a different task you'd choose differently. To find claim numbers (Lab 3) you must **keep** them. That's why the
cleaning is written as your own functions, where every choice is visible, rather than a library default you can't see.

## 2. Cleaning and tokenising two languages at once

Before a model can learn from text, the text has to be cleaned and split into words. Most NLP tutorials assume
English. Here's what changes when your text isn't only English:

- **Stop words** (words too common to carry meaning) need a Swahili list as well: *na, ya, wa, la, kwa, ni...*
  NLTK and scikit-learn only ship English ones.
- **Stemming** tools such as the Porter stemmer are English-only. Applied to Swahili they mangle words, so this lab
  doesn't stem.
- **Structured tokens** such as claim references and member numbers aren't words. Replace them with placeholder words
  (`claimref`, `memberno`) so the model learns "mentions a claim reference" rather than memorising numbers.
  It's also a first step towards not spreading personal data around.

### The stop-word list

Build one list from three parts. Python **sets** make this easy: `-` removes items, `|` combines two sets.

1. scikit-learn's English stop words, **minus** the negations. *not, no, never* are on the standard list, but for
   sentiment they're vital: "paid" and "not paid" mean opposite things;
2. **plus** a hand-made Swahili list;
3. **plus** words that call-centre agents add to every summary (*caller reports*, *mpigaji anasema*), which say nothing
   about the member's problem.

In [ ]:
# Common Swahili words that carry no meaning on their own.
SWAHILI_STOP = {"na", "ya", "wa", "la", "kwa", "ni", "za", "kwenye", "katika", "hii", "hiyo", "huo", "yangu", "wangu",
                "mimi", "sisi", "tu", "pia", "au", "kama", "hata", "sana", "je", "ili", "bado", "tangu", "mpaka", "sasa",
                "lakini", "kwamba", "mwezi", "tafadhali", "habari", "ndugu", "samahani", "asanteni", "asante"}

# Negations are on scikit-learn's English stop list, but they matter for sentiment ("not paid"), so keep them.
KEEP = {"not", "no", "never", "nobody", "nothing", "cannot"}

# The full stop list: English minus the negations, plus Swahili, plus call-centre filler words.
STOP = set(ENGLISH_STOP_WORDS) - KEEP
STOP = STOP | SWAHILI_STOP
STOP = STOP | {"caller", "reports", "mpigaji", "anasema"}
print(len(STOP), "stop words")

### Cleaning, one step at a time

Cleaning in four small steps. Try them on one made-up message first, so you can see what each does.

Steps 2 to 4 use **regular expressions**: small patterns that describe text. `re.sub(pattern, replacement, text)`
finds every piece of text matching the pattern and replaces it. The symbols used here:

| Symbol | Means | Example |
|---|---|---|
| `\d` | any digit 0-9 | `\d\d` matches `42` |
| `{4}` | exactly 4 of the previous thing | `\d{4}` matches `2025` |
| `\b` | a word boundary (start or end of a word) | `\b\d{9}\b` matches a 9-digit number on its own |
| `\w+` | one or more letters, digits or underscores | matches `delayed` |
| `( )` and `\1` | capture a part, then reuse it in the replacement | `#(\w+)` → `\1` keeps the word, drops the `#` |
| `\|` | or | `pls\|plz` matches either |

In [ ]:
text = "Caller reports: my claim CLM-2025-004512 for member 123456789 is #delayed, pls help"

# 1. Lower case.
step1 = text.lower()

# 2. Claim references like clm-2025-004512 become the word "claimref". \d means "a digit", {4} means "exactly 4".
step2 = re.sub(r"clm-\d{4}-\d{6}", " claimref ", step1)

# 3. Any 9-digit number becomes "memberno". \b means "a word boundary", so it must be a whole number.
step3 = re.sub(r"\b\d{9}\b", " memberno ", step2)

# 4. Hashtags keep the word, lose the #. "pls" and "plz" become "please".
step4 = re.sub(r"#(\w+)", r" \1 ", step3)
step4 = re.sub(r"\bpls\b|\bplz\b", "please", step4)

print(step4)

Now **tokenise**: split the cleaned text into pieces with `wordpunct_tokenize`, then keep only the pieces that carry
meaning:

- `isalpha()`: letters only (drops punctuation and numbers);
- `not in STOP`: not a stop word;
- `len(token) > 1`: longer than one letter.

In [ ]:
# Split into words, then keep only real words that aren't stop words and are longer than one letter.
tokens = wordpunct_tokenize(step4)
print("all tokens: ", tokens)

kept = []
for token in tokens:
    if token.isalpha() and token not in STOP and len(token) > 1:
        kept.append(token)
print("kept tokens:", kept)

The same steps as two functions, so you can use them on every message (and so TF-IDF can use them in Section 3):

- `normalise(text)` does the four cleaning steps and returns the cleaned text;
- `tokenise(text)` normalises, splits, and returns the list of meaningful words.

Then test them on a real message from the data.

In [ ]:
def normalise(text):
    """Lower case, and replace claim references and member numbers with placeholder words."""
    text = text.lower()
    text = re.sub(r"clm-\d{4}-\d{6}", " claimref ", text)      # claim references
    text = re.sub(r"\b\d{9}\b", " memberno ", text)            # member numbers
    text = re.sub(r"#(\w+)", r" \1 ", text)                    # hashtags
    text = re.sub(r"\bpls\b|\bplz\b", "please", text)          # common abbreviations
    return text


def tokenise(text):
    """Split a message into the words that carry meaning: letters only, no stop words, longer than one letter."""
    kept = []
    for token in wordpunct_tokenize(normalise(text)):
        if token.isalpha() and token not in STOP and len(token) > 1:
            kept.append(token)
    return kept


example = fb["text"].iloc[3]
print(example)
print(tokenise(example))

Look at the most common tokens overall. It's a quick sanity check that cleaning did what you meant. `counts.update`
adds one message's words to the running totals; `most_common(25)` returns the 25 most frequent.

In [ ]:
counts = Counter()
for message in fb["text"]:
    counts.update(tokenise(message))      # add this message's words to the running counts

# The 25 most common words, shown sideways (.T) to save space.
pd.DataFrame(counts.most_common(25), columns=["token", "count"]).T

**Your turn (1).** The list above still contains words that tell you nothing (look for greetings and filler words in
both languages). Add a few more to `SWAHILI_STOP` or `STOP` and rerun the cells. Don't overdo it: removing
*sijalipwa* ("I haven't been paid") would throw away exactly the signal you want.

> **Where we are.** `tokenise` turns any message into a list of meaningful words in either language, with negations
> kept and claim and member numbers replaced by placeholders. You've checked the most common tokens look sensible.
> Now the words become numbers a model can learn from.

## 3. Sentiment with TF-IDF and logistic regression

A model needs numbers, not words. The simplest idea is to count words: one column per word in the vocabulary, and each
message's row says how often it uses each word. **TF-IDF** improves on plain counting by weighting the counts:

- **TF (term frequency):** how often the word appears in *this* message. More often, more weight.
- **IDF (inverse document frequency):** how rare the word is across *all* messages. A word that's in almost every
  message (*pension* in a pensions complaint channel) tells you little, so it's weighted down.

So a word gets a high TF-IDF score when it appears in this message but is rare across all messages: it's
**distinctive**. With `ngram_range=(1, 2)` it also counts word pairs, so *"not paid"* becomes a feature of its own, not
just *not* and *paid* separately.

### TF-IDF on two tiny messages

Suppose the whole collection were these two messages:

```
A: "pension not paid"
B: "pension paid thank you"
```

*pension* and *paid* appear in both, so they say little about which message is which, and TF-IDF gives them a low weight.
*not* appears only in A, and *thank* only in B, so they get a high weight. Each message becomes a row of numbers, one per
word in the vocabulary, with the distinctive words counting most. That row is what the logistic regression learns from.

`fit_transform` learns the vocabulary and weights, and converts the messages. `toarray()` turns the result into an
ordinary table you can look at; `get_feature_names_out()` gives the column names (the words).

In [ ]:
# See it for yourself on those two messages.
tiny = TfidfVectorizer()
tiny_vectors = tiny.fit_transform(["pension not paid", "pension paid thank you"])
pd.DataFrame(tiny_vectors.toarray(), columns=tiny.get_feature_names_out(), index=["A", "B"]).round(2)

Split first. The vocabulary and the IDF weights are learned from the training messages only, so the test messages are
truly unseen (the same rule as every split since Day 2). `stratify` keeps the same mix of sentiments in both parts.

In [ ]:
# 75% to train, 25% to test, with the same mix of sentiments in both.
train, test = train_test_split(fb, test_size=0.25, random_state=SEED, stratify=fb["sentiment"])
print(len(train), "training messages,", len(test), "test messages")

Now the model: a **pipeline** of two steps. TF-IDF turns each message into numbers using your `tokenise` function;
logistic regression learns which words go with which sentiment. `class_weight="balanced"` gives the rarer sentiments
(positive) more weight, so the model doesn't ignore them.

`classification_report` shows, for each sentiment:

- **precision:** of the messages predicted as this sentiment, the share that really were;
- **recall:** of the messages that really were this sentiment, the share the model found;
- **f1-score:** a balance of the two (1 is perfect);
- **support:** how many test messages had this sentiment.

In [ ]:
# TF-IDF settings:
#   tokenizer=tokenise    use our cleaning function to split messages into words
#   token_pattern=None    (we gave our own tokenizer, so switch off the default one)
#   lowercase=False       our normalise() already lower-cases
#   ngram_range=(1, 2)    single words and word pairs
#   min_df=2              ignore words seen in only one message
sentiment_model = Pipeline([
    ("tfidf", TfidfVectorizer(tokenizer=tokenise, token_pattern=None, lowercase=False, ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
])
sentiment_model.fit(train["text"], train["sentiment"])       # learns vocabulary, weights and model from training only

pred = sentiment_model.predict(test["text"])
print(classification_report(test["sentiment"], pred, digits=3))

### Theory: macro F1, and why it's the headline here

With three classes, there are several ways to boil the results down to one number:

- **accuracy:** the share of messages classified correctly. Dominated by the common class (negative).
- **weighted F1:** each class's F1, averaged in proportion to how many messages it has. Also dominated by negative.
- **macro F1:** each class's F1, averaged **equally**. A rare class (positive) counts as much as the common one.

If the model gets negative messages right and never spots a positive one, accuracy and weighted F1 still look good,
but macro F1 drops sharply. Because the Fund wants to know about all three kinds of message, macro F1 is the fairer
headline.

Look beyond the headline. Positive feedback is rare, so check its recall in particular: a model that never spots a
compliment isn't much use for a "what are members happy about?" report.

### Which words drive each prediction?

Logistic regression learns one **weight** (coefficient) per word per sentiment. A large positive weight means "this
word pushes the prediction towards this sentiment". Because each feature is a word, you can simply read the model's
reasoning, which is a big advantage of classical NLP.

`named_steps` reaches into the pipeline for each step; `classes_` lists the sentiments; `coef_[i]` holds the weights for
sentiment number `i`.

In [ ]:
vocabulary = sentiment_model.named_steps["tfidf"].get_feature_names_out()     # the words (and word pairs)
classifier = sentiment_model.named_steps["clf"]

# One row of weights per sentiment. For each, show the 12 words with the largest weights.
for i in range(len(classifier.classes_)):
    label = classifier.classes_[i]
    weights = pd.Series(classifier.coef_[i], index=vocabulary)
    top_words = weights.sort_values(ascending=False).head(12).index
    print(f"{label:9s}:", ", ".join(top_words))

You should see English and Swahili words side by side, for example *asanteni* next to *thank*. The model didn't
need to be told they mean similar things. It learned from the labels that both show up in positive messages.

### Where does it go wrong?

Reading the mistakes is one of the most useful things you can do with any model. Put each test message's prediction
next to its label, and keep the rows where they differ.

In [ ]:
test = test.copy()           # a separate copy, so adding a column doesn't affect the original split
test["predicted"] = pred

errors = test[test["sentiment"] != test["predicted"]]      # label and prediction disagree
errors[["language", "text", "sentiment", "predicted"]].head(10)

When you read the errors, some will be genuine model mistakes, and some will be **label** mistakes, where the person
who labelled the message got it wrong. A classifier can't be more accurate than its labels. If you find many
labelling errors, the fix is a clearer labelling guide, not a cleverer model.

**Your turn (2).** Does the model work equally well in both languages? Compute the macro F1 score on the test set
separately for `language == "en"`, `"sw"` and `"mixed"`. If one language is clearly worse, what would you do about it?

In [ ]:
# Your code here

> **Where we are.** The TF-IDF model classifies sentiment well, you can read which words drive each prediction, and
> you've looked at its mistakes (some of which are labelling mistakes). Its weakness is that it treats every word as
> unrelated to every other.

## 4. Word embeddings with Word2Vec

TF-IDF treats every word as unrelated to every other: *pension* and *pensheni* are as different as *pension* and
*queue*. **Word embeddings** fix that by giving each word a vector (a list of numbers), learned from the contexts it
appears in. Words used in similar contexts end up close together.

The idea behind it is old and simple: *you can tell a word by the company it keeps*. If *pensheni* keeps appearing
next to *sijalipwa*, *mwezi* and *malipo*, and so does *pension* next to *paid*, *month* and *payment*... the two end up
with similar vectors.

**Word2Vec** learns these vectors by playing a guessing game over and over: given a word, predict the words around it.
To get good at the game it has to give similar vectors to words that appear in similar places.

Pre-trained embeddings (GloVe, fastText) are trained on huge general collections of text and know little Swahili and
nothing about the Fund. Training your own on 6,000 messages is small, but it learns the Fund's vocabulary.

The settings:

| Setting | Meaning |
|---|---|
| `vector_size=64` | each word becomes 64 numbers |
| `window=5` | "nearby" means up to 5 words either side |
| `min_count=3` | ignore words seen fewer than 3 times (too rare to learn) |
| `sg=1` | the "skip-gram" version of the guessing game, which works better for rare words |
| `epochs=30` | go through all the messages 30 times |
| `workers=1`, `seed` | one worker and a fixed seed, so you get the same result every run |

In [ ]:
from gensim.models import Word2Vec

# Word2Vec learns from lists of words: one list per message.
sentences = []
for message in fb["text"]:
    sentences.append(tokenise(message))

# vector_size=64: each word becomes 64 numbers. window=5: look 5 words either side for context.
# min_count=3: ignore words seen fewer than 3 times. workers=1 + seed: the same result every run.
w2v = Word2Vec(sentences, vector_size=64, window=5, min_count=3, sg=1, epochs=30, seed=SEED, workers=1)
print(f"vocabulary: {len(w2v.wv):,} words")

`most_similar` finds the words whose vectors are closest to a given word's vector. The `if` skips any word that didn't
make it into the vocabulary.

In [ ]:
# The 6 words closest to each of these.
for word in ["pension", "pensheni", "app", "employer", "queue"]:
    if word in w2v.wv:
        similar = w2v.wv.most_similar(word, topn=6)       # a list of (word, similarity) pairs
        similar_words = []
        for similar_word, similarity in similar:
            similar_words.append(similar_word)
        print(f"{word:10s} ->", ", ".join(similar_words))

And here's what a word vector actually is: just a list of numbers (the first 10 of 64 shown). On their own they mean
nothing; only the distances between words' vectors carry meaning.

In [ ]:
# One word's vector: just 64 numbers.
w2v.wv["pension"][:10].round(3)

Look at what sits closest to *employer* and *pensheni*: mostly **misspellings** of the same word (*emplyer*,
*eployer*, *penshni*). WhatsApp and social media messages are full of them, and Word2Vec has worked out on its own
that they're used in the same places as the correct spelling. That's genuinely useful: it gives you a list of
spelling variants to normalise, in both languages, without writing it by hand. With a year of real call-centre
transcripts you'd also start to see related *concepts* cluster together, not just spellings.

### Can the embeddings classify sentiment?

To classify a whole message you need one vector per *message*, not per word. The simplest way: take the average of its
words' vectors. A message whose words are all unknown gets 64 zeros.

`np.vstack` stacks the message vectors into a table: one row of 64 numbers per message.

In [ ]:
def message_vector(text):
    """A whole message as the average of its words' Word2Vec vectors (zeros if none of its words are known)."""
    vectors = []
    for token in tokenise(text):
        if token in w2v.wv:                    # only words Word2Vec has learned
            vectors.append(w2v.wv[token])
    if len(vectors) == 0:
        return np.zeros(w2v.vector_size)
    return np.mean(vectors, axis=0)            # the average of the word vectors


# One row of 64 numbers per message.
X_train_w2v = np.vstack(train["text"].map(message_vector))
X_test_w2v = np.vstack(test["text"].map(message_vector))
print(X_train_w2v.shape)

Train a logistic regression on those 64 numbers and compare its macro F1 (the F1 score averaged over the three
sentiments, so the rare positive class counts as much as the common negative one) with the TF-IDF model.

In [ ]:
emb_clf = LogisticRegression(max_iter=3000, class_weight="balanced")
emb_clf.fit(X_train_w2v, train["sentiment"])

f1_w2v = f1_score(test["sentiment"], emb_clf.predict(X_test_w2v), average="macro")
f1_tfidf = f1_score(test["sentiment"], pred, average="macro")
print("macro F1, averaged Word2Vec:", round(f1_w2v, 3))
print("macro F1, TF-IDF:           ", round(f1_tfidf, 3))

Averaging word vectors throws away word order and negation, so on a task like sentiment TF-IDF with word pairs often
wins. Embeddings earn their keep elsewhere: finding similar messages, grouping them, and as the input to deeper
models. Their big limitation is that each word gets **one** vector whatever the sentence. *"Paid"* in "I was paid" and
"I was not paid" looks the same. That's the problem transformers solve, and it's why Lab 2 uses one.

> **Where we are.** Word2Vec has learned the Fund's own vocabulary, including spelling variants in both languages, but
> averaged word vectors classify sentiment less well than TF-IDF. So far everything has used the staff's labels. The
> last technique needs none.

### Theory: supervised classification versus topic discovery

The category labels answer *"which of our nine known kinds of message is this?"*. That's supervised, like Day 2. It
can't tell you about a kind of complaint that has no category yet.

**Topic modelling** is unsupervised, like Day 2's clustering and Day 5's anomaly detection. It finds groups of words
that tend to turn up together, and lets the data suggest the themes. As with clusters, the algorithm finds the groups
and a person names them. Its value is in surprises: a theme that suddenly grows, or one nobody expected.

## 5. What are people complaining about? Topics over time

The customer service team's categories are fixed. Topic modelling finds themes on its own, which is how you notice a
new kind of complaint before anyone has made a category for it.

### How NMF finds topics

Start with the TF-IDF table: one row per message, one column per word. **NMF** (non-negative matrix factorisation)
splits that big table into two smaller ones:

- a **topics × words** table: each topic is a set of word weights (e.g. topic 3 = *app, login, password, otp...*);
- a **messages × topics** table (`W`): how much of each topic is in each message.

Multiply them back together and you get approximately the original table. Words that keep appearing together end up in
the same topic. You choose the number of topics (here 7); NMF finds what they are.

Run it on negative messages only, since those are the ones that need action. `max_df=0.5` ignores words that appear in
more than half the messages (too common to define a topic); `min_df=5` ignores words in fewer than 5.

In [ ]:
neg = fb[fb["sentiment"] == "negative"].copy()

# TF-IDF again. max_df=0.5: ignore words in more than half the messages (too common to define a topic).
topic_tfidf = TfidfVectorizer(tokenizer=tokenise, token_pattern=None, lowercase=False, min_df=5, max_df=0.5)
T = topic_tfidf.fit_transform(neg["text"])

# NMF finds 7 topics. W says how much of each topic is in each message.
nmf = NMF(n_components=7, random_state=SEED, init="nndsvda", max_iter=500)
W = nmf.fit_transform(T)
print(W.shape, "= (messages, topics)")

A topic is just a list of word weights, so to understand it, look at its heaviest words. `nmf.components_[k]` holds
topic `k`'s weight for every word.

In [ ]:
# Each topic is a weight per word. Show the 9 heaviest words of each.
terms = topic_tfidf.get_feature_names_out()

for k in range(7):
    weights = pd.Series(nmf.components_[k], index=terms)
    top_words = weights.sort_values(ascending=False).head(9).index
    print(f"topic {k}:", ", ".join(top_words))

**Your turn (3).** Give each topic a short name, as you did with clusters on Day 2. Replace the placeholders below.
NMF numbers its topics arbitrarily, so the names depend on what *you* see in the lists above.

In [ ]:
# e.g. {0: "delayed benefit", 1: "app / login", ...}
topic_names = {
    0: "topic 0",
    1: "topic 1",
    2: "topic 2",
    3: "topic 3",
    4: "topic 4",
    5: "topic 5",
    6: "topic 6",
}

Now the useful part for management: how each topic's share of negative feedback moves over time.

1. Give each message its strongest topic: `W.argmax(axis=1)` picks, for each row, the column with the largest value;
2. label each message with its quarter (`to_period("Q")`, e.g. 2025Q2);
3. `pd.crosstab` counts messages per quarter and topic; `normalize="index"` turns each row into shares that add up to 1.

In [ ]:
# Each message's strongest topic: the column of W with the largest value.
neg["topic"] = W.argmax(axis=1)
neg["topic_name"] = neg["topic"].map(topic_names)
neg["quarter"] = neg["received_at"].dt.to_period("Q")

# Per quarter, the share of negative messages in each topic (each row adds up to 1).
share = pd.crosstab(neg["quarter"], neg["topic_name"], normalize="index")
share.round(2)

In [ ]:
share.plot(figsize=(11, 4.5), marker="o")      # one line per topic
plt.title("Share of negative feedback by topic, per quarter", loc="left")
plt.ylabel("share")
plt.legend(fontsize=8, ncol=2)
plt.show()

Look for a topic that jumps in one quarter. In this data, one does, in the second quarter of 2025. Find it, read a
handful of the messages behind it, and connect it to what happened. (Hint: the Fund launched a new mobile app in
April 2025.) A chart like this, refreshed monthly, is an early-warning system: it can show a problem with a new system
within weeks of go-live, long before it appears in any formal report.

> **Where we are.** You've found seven themes in negative feedback, named them, and tracked their shares by quarter,
> including the jump after the new app's launch.

## 6. Wrap-up

| Technique | Good at | Watch out for |
|---|---|---|
| TF-IDF + logistic regression | Fast, strong baseline, explainable words | No sense of meaning or word order beyond pairs |
| Word2Vec | Similar words, similar messages, compact vectors | One vector per word regardless of context |
| NMF topics | Discovering themes without labels | Topics need a human to name and interpret |

Save the sentiment model for later days. Because it's a pipeline, the saved file contains the TF-IDF step too, so it
can be given raw text directly. (Your `tokenise` function must exist in any notebook that loads it.)

In [ ]:
joblib.dump(sentiment_model, "sentiment_tfidf.joblib")
print("saved")

---

## Looking back, looking ahead

**What you did in this lab.** You cleaned and tokenised bilingual text with choices made for the task, built a TF-IDF
and logistic-regression sentiment model you can read word by word, checked its mistakes and its quality by language,
learned word embeddings from the Fund's own messages, and used NMF to find complaint themes and watch them over time.

**The ideas to carry forward.**

- Every cleaning step throws something away; choose for the task (keep the negations).
- Start with TF-IDF and a linear model: fast, strong and explainable.
- Read the mistakes; some are the labels' fault.
- Check quality by language, not just on average.
- Topics, like clusters, need a person to give them meaning.

**What happens next.** Lab 2 uses a transformer, a model that gives each word a vector that depends on its sentence, to
route complaints to the right team.

## Suggested answers

**(2)** F1 by language. Filter the test set to one language at a time and score the predictions already made:

In [ ]:
for language in ["en", "sw", "mixed"]:
    in_language = test[test["language"] == language]
    f1 = f1_score(in_language["sentiment"], in_language["predicted"], average="macro")
    print(f"{language:6s} n={len(in_language):4d}  macro F1 {f1:.3f}")

If Swahili is weaker, the usual fixes in order of cost: more labelled Swahili examples, a Swahili stop-word and
spelling-variant list, and a multilingual model (Lab 2). Report quality by language as a matter of routine. A model
that works well "on average" can still be failing a large group of members.

**(3)** Topic names depend on your run. Typical ones here: *delayed benefit payments*, *contributions missing from
statements*, *employer not remitting*, *app, portal and USSD problems*, *queues and staff attitude*, *pension and
verification*, *registration details*.